# street_crime Table Cleaning

pre-step: checks how many values are missing; checks distribution of crime types
1. Drops all rows with missing lsoa_code
2. Drops Welsh crime data from the table
3. Groups crime types together based on plan
4. Checks for missing LSOAs, maps to police force areas, shows map of UK and the two police force areas with missing LSOAs
5. Creates new table lsoa_month

In [ ]:
import geopandas as gpd
import sqlite3
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import libpysal
import seaborn as sns

conn = sqlite3.connect('../data/police_data.db')
cursor = conn.cursor()

In [ ]:
#pre-step; check how many values missing, check distribution of crime types
total_rows = pd.read_sql("SELECT COUNT(*) as total FROM street_crimes;", conn).iloc[0,0]
print(f'total rows {total_rows}')

null_query = """
SELECT 
    COUNT(*) - COUNT(crime_id) as missing_crime_id,
    COUNT(*) - COUNT(month) as missing_month,
    COUNT(*) - COUNT(longitude) as missing_long,
    COUNT(*) - COUNT(latitude) as missing_lat,
    COUNT(*) - COUNT(lsoa_code) as missing_lsoa,
    COUNT(*) - COUNT(crime_type) as missing_crime,
    COUNT(*) - COUNT(last_outcome) as missing_outcome
FROM street_crimes;
"""
null_df = pd.read_sql(null_query, conn)
null_percentages = (null_df / total_rows) * 100
display(null_percentages.round(2).astype(str) + '%')

crime_type_query = """
SELECT crime_type, COUNT(*) as incident_count 
FROM street_crimes 
GROUP BY crime_type 
ORDER BY incident_count DESC;
"""
crime_df = pd.read_sql(crime_type_query, conn)
crime_df['percentage'] = ((crime_df['incident_count'] / total_rows) * 100).round(2)
display(crime_df)

In [ ]:
#1 dropping all rows that have no lsoa_code in street_crimes

#count the rows before deletion
cursor.execute('SELECT COUNT(*) FROM street_crimes;')
count_before = cursor.fetchone()
print(f'count before deletion {count_before}')

#deletes all rows where lsoa_code is empty
delete = 'DELETE FROM street_crimes WHERE lsoa_code IS NULL;'
cursor.execute(delete)
conn.commit()

#count the rows after deletion
cursor.execute("SELECT COUNT(*) FROM street_crimes;")
count_after = cursor.fetchone()
print(f'count after deletion {count_before}')

In [ ]:
#2 drop all Welsh crime from street_crimes tables

#checking how many rows before deleting 
cursor.execute("SELECT COUNT(*) FROM street_crimes;")
count_before = cursor.fetchone()[0]
print(f'count before {count_before}')

#deleting w cursor, W% deletes any row where lsoa_code begins with W (welsh)
cursor.execute("DELETE FROM street_crimes WHERE lsoa_code LIKE 'W%';")

#commiting to db
conn.commit()

#checking how many rows after
cursor.execute("SELECT COUNT(*) FROM street_crimes;")
count_after = cursor.fetchone()[0]
print(f'count after {count_after}')

#difference
print(count_before - count_after)

In [ ]:
#3 grouping crime types together

#first deleting all other crime
cursor.execute("DELETE FROM street_crimes WHERE crime_type IN ('Other crime');")

#grouping violence
cursor.execute("""
    UPDATE street_crimes 
    SET crime_type = 'Violence' 
    WHERE crime_type IN ('Violence and sexual offences', 'Violent crime', 'Robbery');""")

#grouping property theft
cursor.execute("""
    UPDATE street_crimes 
    SET crime_type = 'Property theft' 
    WHERE crime_type IN ('Shoplifting', 'Theft from the person', 'Bicycle theft', 'Other theft');""")

#grouping disruptive crimes
cursor.execute("""
    UPDATE street_crimes 
    SET crime_type = 'Disruptive crimes' 
    WHERE crime_type IN ('Drugs', 'Possession of weapons', 'Public disorder and weapons', 'Public order');""")

conn.commit()

In [ ]:
#3a check missing etc now that we've cleaned street_crimes
total_rows = pd.read_sql("SELECT COUNT(*) as total FROM street_crimes;", conn).iloc[0,0]
print(f'total rows {total_rows}')

null_query = """
SELECT 
    COUNT(*) - COUNT(crime_id) as missing_crime_id,
    COUNT(*) - COUNT(month) as missing_month,
    COUNT(*) - COUNT(longitude) as missing_long,
    COUNT(*) - COUNT(latitude) as missing_lat,
    COUNT(*) - COUNT(lsoa_code) as missing_lsoa,
    COUNT(*) - COUNT(crime_type) as missing_crime,
    COUNT(*) - COUNT(last_outcome) as missing_outcome
FROM street_crimes;
"""
null_df = pd.read_sql(null_query, conn)
null_percentages = (null_df / total_rows) * 100
display(null_percentages.round(2).astype(str) + '%')

crime_type_query = """
SELECT crime_type, COUNT(*) as incident_count 
FROM street_crimes 
GROUP BY crime_type 
ORDER BY incident_count DESC;
"""
crime_df = pd.read_sql(crime_type_query, conn)
crime_df['percentage'] = ((crime_df['incident_count'] / total_rows) * 100).round(2)
display(crime_df)

In [ ]:
#4 checking for missing lsoas in street_crimes

#clean data gathered
clean_lsoas_df = pd.read_csv("../data/lsoa_codes.csv", usecols=['LSOA21CD', 'LSOA21NM'])
total_lsoa_set = set(clean_lsoas_df['LSOA21CD'])

#get from db all lsoas
db_lsoas_df = pd.read_sql("SELECT lsoa_code FROM street_crimes;",  conn)

#i use set here so there are no duplicates
db_lsoa_set = set(db_lsoas_df['lsoa_code'])

#set difference to find which lsoa's missing from our db
missing_lsoas = total_lsoa_set - db_lsoa_set

print(f'total lsoa codes count (clean) {len(total_lsoa_set)}')
print(f'database lsoa codes count {len(db_lsoa_set)}')
print(f'number of missing lsoas {len(missing_lsoas)}')

#convert missing lsoas to series
missing_series = pd.Series(list(missing_lsoas))

#take the first letter in each code and count
code_missing = missing_series.str[0].value_counts()

print(f'missing lsoas come from {code_missing}')

#filtering the series to keep rows where it doesn't start with W (~ = not)
missing_england = missing_series[~missing_series.str.startswith('W')]
#print in list format for cleanliness
print(missing_england.tolist())

In [ ]:
#4a plot missing lsoas
gdf = gpd.read_file('../data/lsoa_spatial.geojson') 

missing_codes_list = missing_england.tolist()

#separate between missing and normal lsoas
england_base_map = gdf[~gdf['LSOA21CD'].str.startswith('W', na=False)]
missing_map_df = gdf[gdf['LSOA21CD'].isin(missing_codes_list)]

fig, ax = plt.subplots(1, 1, figsize=(15, 15))

england_base_map.plot(
    ax=ax, 
    color='whitesmoke', 
    edgecolor='lightgrey', 
    linewidth=0.2
)


if not missing_map_df.empty:
    missing_map_df.plot(
        ax=ax, 
        color='red', 
        edgecolor='darkred',
        linewidth=1
    )


ax.axis('off')

plt.savefig('../data/missing_lsoas_map.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#4b map lsoa to police force area
#take all lsoa_code and pfa
df_info = pd.read_sql("SELECT lsoa_code, pfa_name FROM lsoa_info;", conn)

#filter df to only keep rows that have the missing codes
missing_auth_df = df_info[df_info['lsoa_code'].isin(missing_codes_list)]

#count how many codes belong to each pfa
auth_summary = missing_auth_df['pfa_name'].value_counts()

display(auth_summary.to_frame)

#take all lsoa_code and local authority names
df_info = pd.read_sql("SELECT lsoa_code, pfa_name FROM lsoa_info;", conn)

#filtering for normal english lsoas; code can't be in missing code list
normal_auth_df = df_info[(~df_info['lsoa_code'].isin(missing_codes_list))]

#count
normal_summary = normal_auth_df['pfa_name'].value_counts()

#don't hide rows
pd.set_option('display.max_rows', None)

display(normal_summary.to_frame())

In [ ]:
#4c check populations of missing lsoas

#join to get lsoa_info and lsoa_demographics together
query = """
SELECT 
    i.lsoa_code, 
    i.pfa_name, 
    d.pop as population
FROM lsoa_info i
LEFT JOIN lsoa_demographics d ON i.lsoa_code = d.lsoa_code;
"""
df_combined = pd.read_sql(query, conn)

#filter for only missing English LSOAs
missing_df = df_combined[df_combined['lsoa_code'].isin(missing_codes_list)]

display(missing_df.sort_values('population', ascending=True))

In [ ]:
#4d make an in depth map of the two areas w/ missing lsoas
df_info = pd.read_sql("SELECT lsoa_code, pfa_name FROM lsoa_info;", conn)

gdf = gpd.read_file('../data/lsoa_spatial.geojson')

#merge shapes w/ police force names
map_with_police = gdf.merge(df_info, left_on='LSOA21CD', right_on='lsoa_code', how='inner')

#base maps
london_base = map_with_police[map_with_police['pfa_name'] == 'Metropolitan Police']
manchester_base = map_with_police[map_with_police['pfa_name'] == 'Greater Manchester']

#grab the specific missing lsoas
london_missing = london_base[london_base['LSOA21CD'].isin(missing_codes_list)]
manchester_missing = manchester_base[manchester_base['LSOA21CD'].isin(missing_codes_list)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

#london map
london_base.plot(ax=ax1, color='whitesmoke', edgecolor='lightgrey', linewidth=0.5)
if not london_missing.empty:
    london_missing.plot(ax=ax1, color='red', edgecolor='darkred', linewidth=1.5)
ax1.axis('off')
ax1.set_title("Missing LSOAs: London (Metropolitan Police)", fontsize=18, pad=15)

#machester map
manchester_base.plot(ax=ax2, color='whitesmoke', edgecolor='lightgrey', linewidth=0.5)
if not manchester_missing.empty:
    manchester_missing.plot(ax=ax2, color='red', edgecolor='darkred', linewidth=1.5)
ax2.axis('off')
ax2.set_title("Missing LSOAs: Greater Manchester", fontsize=18, pad=15)

# 7. Save and show
plt.tight_layout()
plt.savefig('../data/missing_indepth_map.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#5 create new table lsoa_month

#deletes table if exists to let me re-run multiple times
cursor.execute("DROP TABLE IF EXISTS lsoa_month;")

create_table_query = """
CREATE TABLE lsoa_month (
    lsoa_code TEXT,
    month TEXT,
    crime_type TEXT,
    crime_count INTEGER,
    PRIMARY KEY (lsoa_code, month, crime_type)
);
"""
cursor.execute(create_table_query)


#zero-padding aggregation query
aggregate_insert_query = """
INSERT INTO lsoa_month (lsoa_code, month, crime_type, crime_count)

WITH 
    unique_lsoas AS (SELECT DISTINCT lsoa_code FROM street_crimes),
    unique_months AS (SELECT DISTINCT month FROM street_crimes),
    unique_crimes AS (SELECT DISTINCT crime_type FROM street_crimes),
    
    master_grid AS (
        SELECT l.lsoa_code, m.month, c.crime_type
        FROM unique_lsoas l
        CROSS JOIN unique_months m
        CROSS JOIN unique_crimes c
    ),
    
    actual_crimes AS (
        SELECT lsoa_code, month, crime_type, COUNT(*) as count
        FROM street_crimes
        GROUP BY lsoa_code, month, crime_type
    )

SELECT 
    g.lsoa_code, 
    g.month, 
    g.crime_type, 
    COALESCE(a.count, 0) as crime_count
FROM master_grid g
LEFT JOIN actual_crimes a 
    ON g.lsoa_code = a.lsoa_code 
    AND g.month = a.month 
    AND g.crime_type = a.crime_type;
"""
cursor.execute(aggregate_insert_query)
conn.commit()

#check
check_query = "SELECT * FROM lsoa_month LIMIT 10;"
check_df = pd.read_sql(check_query, conn)
display(check_df)

total_rows = pd.read_sql("SELECT COUNT(*) as total FROM lsoa_month;", conn).iloc[0,0]
print(total_rows)

In [ ]:
zero_query = "SELECT COUNT(*) FROM lsoa_month WHERE crime_count = 0;"

zero_count = pd.read_sql(zero_query, conn).iloc[0, 0]
print(zero_count)